# AI-Based Construction Delay Prediction & Risk Analytics

## Project Overview
This project demonstrates how machine learning can be used to predict whether a construction activity is likely to be delayed.

**Important:** The dataset is synthetic and created for educational/demo purposes. It does not represent actual L&T or any real construction project.

### Business Problem
Construction delays can increase project cost and affect downstream activities. The objective is to identify high-risk activities early so project teams can take preventive action.

### ML Approach
1. Load and explore historical-style construction data.
2. Preprocess categorical and numerical variables.
3. Train Logistic Regression and Random Forest models.
4. Compare model performance.
5. Identify important risk drivers.
6. Generate delay-risk predictions for new activities.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

df = pd.read_csv("construction_data.csv")
df.head()


In [ ]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["delay"].value_counts(normalize=True))


In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(data=df, x="delay")
plt.title("Distribution of Construction Delays")
plt.xlabel("Delay (0 = No, 1 = Yes)")
plt.ylabel("Number of Activities")
plt.show()


In [ ]:
numeric_cols = [
    "material_availability_pct", "labour_availability_pct",
    "equipment_availability_pct", "vendor_performance_pct",
    "weather_severity", "activity_complexity",
    "planned_duration_days", "previous_delays", "change_orders"
]

plt.figure(figsize=(10,7))
sns.heatmap(df[numeric_cols + ["delay"]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()


In [ ]:
X = df.drop(columns="delay")
y = df["delay"]

categorical_cols = ["activity_type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("numeric", StandardScaler(), [c for c in X.columns if c not in categorical_cols])
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)


In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]


In [ ]:
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]


In [ ]:
def evaluate_model(name, y_true, pred, prob):
    print(f"--- {name} ---")
    print("Accuracy :", round(accuracy_score(y_true, pred), 3))
    print("Precision:", round(precision_score(y_true, pred), 3))
    print("Recall   :", round(recall_score(y_true, pred), 3))
    print("F1 Score :", round(f1_score(y_true, pred), 3))
    print("ROC-AUC  :", round(roc_auc_score(y_true, prob), 3))
    print()

evaluate_model("Logistic Regression", y_test, logistic_pred, logistic_prob)
evaluate_model("Random Forest", y_test, rf_pred, rf_prob)


In [ ]:
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Feature importance from Random Forest
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.named_steps["model"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).head(15)

plt.figure(figsize=(9,6))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Top Factors Influencing Delay Prediction")
plt.show()

importance_df


## Using the Model for a New Activity

The model can be used as an early-warning mechanism. For example, a project manager can enter expected material availability, labour availability, vendor performance, weather severity and other activity attributes to estimate the probability of delay.

A high predicted probability should trigger preventive actions such as expediting material, reallocating labour, checking vendor commitments, or resequencing dependent activities.


In [ ]:
new_activity = pd.DataFrame([{
    "activity_type": "Electrical",
    "material_availability_pct": 68,
    "labour_availability_pct": 72,
    "equipment_availability_pct": 80,
    "vendor_performance_pct": 65,
    "weather_severity": 6,
    "activity_complexity": 4,
    "planned_duration_days": 35,
    "previous_delays": 2,
    "change_orders": 1
}])

risk = rf_model.predict_proba(new_activity)[0, 1]
prediction = rf_model.predict(new_activity)[0]

print(f"Predicted delay probability: {risk:.1%}")
print("Risk classification:", "HIGH RISK" if risk >= 0.60 else "LOW/MODERATE RISK")


## Business Interpretation

The model is intended to support—not replace—project management decisions.

Potential actions based on high-risk predictions:
- **Low material availability:** expedite procurement and identify alternate suppliers.
- **Low labour availability:** rebalance manpower across activities.
- **Poor vendor performance:** initiate vendor escalation and recovery plans.
- **High complexity:** increase supervision and break work into smaller milestones.
- **High weather severity:** resequence weather-sensitive activities.
- **Previous delays/change orders:** create additional schedule buffers and monitor dependencies.

## Future Enhancements
- Integrate real project ERP/BIM/schedule data.
- Add cost overrun prediction.
- Build a Power BI/Tableau dashboard.
- Use XGBoost/LightGBM for additional model comparison.
- Add explainability using SHAP.
- Deploy the model through a Streamlit application.
